# RE_206 · És just el model?

**Bloc 2 · Supervisat · repte 6 de 6 · durada: 2 sessions de 2 h**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes en una consultora que **audita algorismes**. Un banc vol fer servir un model per
**preaprovar** una targeta *premium* a les persones que guanyen més de 50.000 $ l'any, i abans
d'engegar-lo ha de demostrar que el model no discrimina. T'han passat les dades del cens amb què
l'entrenarien.

La pregunta: **el model encerta i s'equivoca igual per a dones i homes, i per als diferents grups? I si
traiem `sex` i `race` de les dades, el problema desapareix?**

> Aquest repte no té una resposta tècnica única. Hi ha números que has de calcular bé, i decisions que
> has de **justificar**.

L'enunciat complet, amb el diccionari de columnes, és a [RE_206](../RE_206_equitat_adult.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

## Què necessites saber abans

- Matriu de confusió, *recall*, *precision* i `DummyClassifier`: [EX_00 §7](../../02_practica/EX_00_caixa_eines.ipynb)
- Nuls, `SimpleImputer`, `OneHotEncoder`, `ColumnTransformer`: [EX_00 §3, §4 i §6](../../02_practica/EX_00_caixa_eines.ipynb)
- Pesos d'una regressió logística: [ML_04](../../01_teoria/ML_04_regressio_logistica.ipynb) i el [RE_202](../RE_202_compres_en_linia.ipynb)
- Llindar de decisió: [EX_02 exercici 7](../../02_practica/EX_02_cancer.ipynb) i el [RE_203](../RE_203_campanya_banc.ipynb)

## Les dades

**Adult / Census Income** (Becker i Kohavi, 1996; UCI), a OpenML amb `data_id=1590`. Llicència CC BY
4.0. Cada fila és **una persona** del cens dels Estats Units de 1994 (anonimitzada): edat, tipus de
feina, estudis, estat civil, ocupació, relació dins de la llar, grup ètnic (`race`, amb les categories
del cens americà), sexe, guanys i pèrdues de capital, hores de feina setmanals i país d'origen. La
columna `class` diu si guanyava més de 50.000 $ l'any.

Documentació: <https://archive.ics.uci.edu/dataset/2/adult>. Llegeix què és `fnlwgt` abans de fer-la
servir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=1590, as_frame=True).frame
print(df.shape)
df.head()

## Nivell 1 · Explorar

### 1.1 · Inventari

Forma, nuls per columna, duplicats i proporció de persones amb `>50K`. Decideix què fas amb els nuls.

In [ ]:
print(df.isna().sum()[df.isna().sum() > 0])
print("duplicats:", df.duplicated().sum())
print(df["class"].value_counts(normalize=True).round(4))
y = (df["class"] == ">50K").astype(int)
assert len(y) == 48842 and set(y.unique()) == {0, 1}

> **Resultat.** Nuls a `workclass` (2.799), `occupation` (2.809) i `native-country` (857), tots de text: es poden tractar com a categoria «desconegut» dins del pipeline. 52 duplicats. Un 23,9 % guanya més de 50K.

### 1.2 · Les dades ja són desiguals?

Abans de cap model: quina proporció de `>50K` hi ha per `sex`? I per `race`? Quantes persones hi ha a cada grup? Per què importa la mida dels grups quan després comparis mesures?

In [ ]:
for g in ["sex", "race"]:
    t = pd.crosstab(df[g], df["class"], normalize="index")
    assert np.allclose(t.sum(axis=1), 1)
    t["n"] = df[g].value_counts()
    assert t["n"].sum() == len(df)
    print(t.round(3), "\n")

> **Resultat.** A les dades, el 30,4 % dels homes i el 10,9 % de les dones guanyen més de 50K. Per grups ètnics també hi ha diferències, i alguns grups són molt petits (`Other` 406 persones, `Amer-Indian-Eskimo` 470): les seves mesures tindran molt soroll. El model aprendrà d'unes dades que ja són desiguals.

### 1.3 · Columnes que diuen el mateix, i una que no és el que sembla

`education` i `education-num`: diuen el mateix? Demostra-ho amb codi. Llegeix a la documentació què és `fnlwgt`: és una característica de la persona? Decideix quines columnes entraran al model.

In [ ]:
n = df.groupby("education", observed=True)["education-num"].nunique()
print(n.max(), "valor d'education-num per a cada education")
print(df.groupby("education", observed=True)["education-num"].first().sort_values())

> **Resultat.** Correspondència exacta d'un a un: `education-num` és l'ordre dels nivells. N'hi ha prou amb una (la numèrica conserva l'ordre). `fnlwgt` és un **pes de mostreig** del cens (a quantes persones representa la fila), no una característica: no ha d'entrar com a atribut.

## Nivell 2 · Respondre

### 2.1 · El model a auditar

Parteix (25 %, estratificat, `random_state=42`) i **guarda a part** `sex` i `race` del test per a l'auditoria. Construeix un pipeline amb un `ColumnTransformer` (imputació i one-hot per a les de text, escalat per a les numèriques) i una regressió logística, amb les columnes que has decidit a l'1.3 (de moment, **amb** `sex` i `race`). Compara'n la precisió de test amb la del `DummyClassifier`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, recall_score, accuracy_score

X = df.drop(columns=["class", "fnlwgt", "education"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
grups_test = X_test[["sex", "race"]].astype(str).copy()
assert len(grups_test) == len(y_test)

def munta(columnes):
    num = [c for c in columnes if pd.api.types.is_numeric_dtype(X[c])]
    txt = [c for c in columnes if c not in num]
    a_text = make_pipeline(SimpleImputer(strategy="constant", fill_value="desconegut"),
                           OneHotEncoder(handle_unknown="ignore"))
    pre = ColumnTransformer([("num", StandardScaler(), num), ("txt", a_text, txt)])
    return make_pipeline(pre, LogisticRegression(max_iter=2000))

def text(Xp):                          # categòriques a text perquè l'imputador les accepti
    Xp = Xp.copy()
    for c in Xp.columns:
        if not pd.api.types.is_numeric_dtype(Xp[c]):
            Xp[c] = Xp[c].astype(object)
    return Xp

cols = list(X.columns)
model = munta(cols).fit(text(X_train), y_train)
Xt = model[0].transform(text(X_test))
assert not np.isnan(Xt.toarray() if hasattr(Xt, "toarray") else Xt).any()
pred = model.predict(text(X_test))
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("precisió model:", round(accuracy_score(y_test, pred), 4), "| dummy:", round(dummy.score(X_test, y_test), 4))

> **Resultat.** La logística fa 0,855 de precisió contra 0,761 del dummy. És un model «bo» en global, que és justament el que no ens diu res sobre l'equitat.

### 2.2 · La teoria, al codi: les mesures per sexe

Escriu una funció que, per a cada grup d'una columna, calculi **a partir de la seva matriu de confusió** el nombre de persones, la precisió, la **taxa de selecció**, la **TPR** (*recall*) i la **FPR**. Aplica-la per `sex`. Comprova la TPR de cada grup amb `recall_score` sobre les files d'aquell grup. Quines diferències hi veus? Quina mesura et sembla més important per a un banc que dona targetes, i per a la persona a qui la deneguen?

In [ ]:
def per_grup(y_true, y_pred, grups):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    files = {}
    for g in sorted(grups.unique()):
        m = (grups == g).values
        tn, fp, fn, tp = confusion_matrix(y_true[m], y_pred[m], labels=[0, 1]).ravel()
        n = m.sum()
        assert tn + fp + fn + tp == n
        tpr = tp / (tp + fn) if tp + fn else np.nan
        assert np.isnan(tpr) or np.isclose(tpr, recall_score(y_true[m], y_pred[m]))
        files[g] = dict(n=n, precisio=(tp + tn) / n, seleccio=(tp + fp) / n, TPR=tpr,
                        FPR=fp / (fp + tn) if fp + tn else np.nan, real_pos=(tp + fn) / n)
    t = pd.DataFrame(files).T
    assert t["n"].sum() == len(y_true)
    return t

t_sex = per_grup(y_test, pred, grups_test["sex"])
t_sex.round(3)

> **Resultat.** Precisió més alta per a les dones (0,926 contra 0,819), però és perquè el «no» és més fàcil d'encertar en un grup on la majoria no arriba a 50K. Taxa de selecció: 0,075 dones, 0,249 homes (a les dades reals, 0,113 i 0,302). TPR: 0,506 contra 0,613; FPR: 0,020 contra 0,092. És a dir: de les dones que **sí** guanyen més de 50K, el model en troba una proporció menor que d'homes. Una precisió millor no vol dir un tracte igual.

### 2.3 · I per grups ètnics

Aplica la mateixa funció a `race`. Què canvia respecte de `sex`? Quins grups tenen tan poques persones al test que les seves mesures no són fiables? Com ho diries a l'informe sense exagerar ni amagar res?

In [ ]:
t_race = per_grup(y_test, pred, grups_test["race"])
t_race["positius_reals"] = (t_race["n"] * t_race["real_pos"]).round().astype(int)
t_race.round(3)

> **Resultat.** TPR: `White` 0,606 i `Asian-Pac-Islander` 0,618, `Black` 0,493, `Other` 0,273 i `Amer-Indian-Eskimo` 0,222. Però aquests dos últims tenen 11 i 18 positius reals al test: un o dos casos mouen la TPR 5-10 punts. La diferència de `Black` (152 positius) sí que és sòlida. L'informe ha de donar els números amb la mida del grup al costat.

### 2.4 · I si traiem `sex` i `race`?

Entrena el mateix pipeline **sense** `sex` ni `race` (però continua auditant amb `grups_test`). Torna a fer les taules per sexe i per grup ètnic. La diferència desapareix? Quant canvia? Per què creus que passa?

In [ ]:
cols2 = [c for c in cols if c not in ("sex", "race")]
model2 = munta(cols2).fit(text(X_train[cols2]), y_train)
noms2 = model2[0].get_feature_names_out()
assert not any(("sex" in s) or ("race" in s) for s in noms2)
pred2 = model2.predict(text(X_test[cols2]))
print("precisió sense sex/race:", round(accuracy_score(y_test, pred2), 4))
t_sex2 = per_grup(y_test, pred2, grups_test["sex"])
assert (t_sex2["n"] == t_sex["n"]).all()
pd.concat({"amb sex/race": t_sex[["seleccio", "TPR", "FPR"]], "sense": t_sex2[["seleccio", "TPR", "FPR"]]}, axis=1).round(3)

> **Resultat.** La precisió global gairebé no canvia (0,8537), i la diferència entre dones i homes **no desapareix**: TPR 0,525 contra 0,604 (abans 0,506 contra 0,613), selecció 0,078 contra 0,246. El model troba el sexe per altres camins (sobretot `relationship`, amb `Husband` i `Wife`, i `marital-status`). Treure l'atribut protegit no és fer el model just: es diu *fairness through unawareness*, i no funciona quan hi ha variables intermediàries.

## Nivell 3 · Més enllà

### 3.1 · Les variables que delaten

Entrena un model que intenti **predir `sex`** a partir de les altres columnes (sense `sex`, `race` ni `class`). Quina precisió treu, comparada amb un dummy? Quines columnes hi pesen més? Què vol dir això per a la pregunta de l'2.4?

In [ ]:
ys_tr = (X_train["sex"].astype(str) == "Female").astype(int)
ys_te = (X_test["sex"].astype(str) == "Female").astype(int)
ms = munta(cols2).fit(text(X_train[cols2]), ys_tr)
print("predir sex -> precisió:", round(ms.score(text(X_test[cols2]), ys_te), 4),
      "| dummy:", round(max(ys_te.mean(), 1 - ys_te.mean()), 4))
pes = pd.Series(ms[-1].coef_.ravel(), index=ms[0].get_feature_names_out())
print(pes.abs().sort_values(ascending=False).head(6).index.tolist())

> **Resultat.** El sexe es prediu amb 0,846 de precisió (dummy 0,671), i les columnes que hi pesen més són `relationship_Husband`, `relationship_Wife` i diverses d'`occupation`. Per això treure `sex` no l'elimina del model.

### 3.2 · Un llindar per grup

Busca un llindar per a les dones i un altre per als homes que facin que la **TPR** sigui gairebé igual en els dos grups (fes servir el model de l'2.4). Quant canvia la precisió global? És una solució justa? Qui hi guanya i qui hi perd? (Aquesta pregunta no té una resposta tècnica: argumenta-la.)

In [ ]:
p2 = model2.predict_proba(text(X_test[cols2]))[:, 1]
dona = (grups_test["sex"] == "Female").values
tpr_h = per_grup(y_test, (p2 >= 0.5).astype(int), grups_test["sex"]).loc["Male", "TPR"]
millor = None
for ll_d in np.arange(0.05, 0.5, 0.01):
    pr = np.where(dona, p2 >= ll_d, p2 >= 0.5).astype(int)
    tpr_d = per_grup(y_test, pr, grups_test["sex"]).loc["Female", "TPR"]
    if millor is None or abs(tpr_d - tpr_h) < millor[1]:
        millor = (ll_d, abs(tpr_d - tpr_h), pr)
ll_d, _, pr = millor
print(f"llindar dones {ll_d:.2f}, homes 0.50 | precisió global {accuracy_score(y_test, pr):.4f} (abans {accuracy_score(y_test, pred2):.4f})")
per_grup(y_test, pr, grups_test["sex"])[["seleccio", "TPR", "FPR"]].round(3)

> **Resultat.** Amb llindar 0,36 per a les dones (0,50 per als homes) la TPR queda igualada (0,605 contra 0,604), la precisió global baixa poc (0,8537 → 0,8494) i la FPR de les dones passa de 0,021 a 0,046. És una decisió de política, no tècnica (i en alguns països, tractar els grups de manera diferent explícitament és il·legal). El que compta és l'argument.

## Conclusió

Hem auditat una regressió logística que encerta el 85,5 % (el model que diu sempre «no», el
76,1 %). El model no tracta igual dones i homes: de les persones que sí que guanyen més de 50K, en
troba el 50,6 % de les dones i el 61,3 % dels homes. Treure `sex` i `race` gairebé no canvia res
(TPR 0,525 contra 0,604), perquè el sexe es dedueix d'altres columnes, sobretot de `relationship`
(es prediu amb un 85 % d'encert). Per grups ètnics, la diferència del grup `Black` (TPR 0,49 contra
0,61 de `White`) és sòlida, però `Other` i `Amer-Indian-Eskimo` tenen massa pocs casos per treure'n
conclusions. Recomanació: no engegar el model sense decidir quina mesura d'equitat s'exigeix,
mesurar-la per grups amb la mida al costat, i no confiar que amagar l'atribut protegit ho resol.